### DeepSearch project

In [23]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown
from messenger import send_email, push
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
from agents import set_default_openai_client
from openai import AsyncOpenAI
from agents import result

In [24]:
load_dotenv(override=True)

True

In [25]:
MODEL_NAME = "gemini-3.6-flash"
USE_EMAIL = True
HOW_MANY_SEARCHES = 5

## Strategy for the Deep Research Agent

We are going to orchestrate with code: separate calls to `Runner.run()` for each step in the process.


## We will build 4 Agents:

### 1. The Search Agent: searches the web for information

2. The Planner Agent: given a question, comes up with a list of searches that should be made
3. The Writer Agent: writes a robust report
4. The Emailer Agent: crafts and sends an email

And then 4 python functions, 1 to call Runner.run() for each of the 4 agents.

# OpenAI offers the following hosted tools:

`WebSearchTool` lets an agent search the web.  
`FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
`CodeInterpreterTool` lets the LLM execute code in a sandboxed environment.  
`HostedMCPTool` exposes a remote MCP server's tools to the model.  
`ImageGenerationTool` generates images from a prompt.  
`ToolSearchTool` lets the model load deferred tools, namespaces, or hosted MCP servers on demand.  

### Important note - API charge of WebSearchTool

# We are using free one thats why we impliment 'duckduckgo'

uv add duckduckgo-search

# creating a tool using duckduckgo

In [26]:
from ddgs import DDGS

@function_tool
def search_web(query: str, max_result: int):
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_result= max_result))
        print(f"websearch is active............ {query}")
    return results


In [27]:

INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
task = "Most popular AI Agent frameworks in 2026"

settings = ModelSettings(tool_choice="required")

#tools = [WebSearchTool()]
tools = [search_web]


In [ ]:
client = AsyncOpenAI(
    #api_key=,
    base_url="https://openrouter.ai/api/v1",
)

set_default_openai_client(client)

MODEL_NAME = "dots-3-note-preview:free"


search_agent = Agent(
    name="Search Agent",
    instructions=INSTRUCTIONS,
    model=MODEL_NAME,
    model_settings=settings,
    tools=tools

)

result = await Runner.run(
    search_agent,
    task
)

print(result.final_output)

#here we get out from web search throuch free Host tool called duckduckgo

[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.


websearch is active............ most popular AI Agent frameworks 2026


[non-fatal] Tracing client error 401. Response data is redacted.




The most popular AI Agent frameworks in 2026 include **LangGraph**, **CrewAI**, **Microsoft Agent Framework (MAF)**, **OpenAI Agents SDK**, **Google ADK**, and **Claude Agent SDK**. These frameworks dominate the landscape due to their production readiness, orchestration capabilities, and strong community adoption. LangGraph and CrewAI are especially prominent, often cited for their flexibility in building multi-agent systems and complex workflows. OpenAI Agents and Google ADK benefit from first-party support and tight integration with their respective ecosystems, while Microsoft's MAF (formerly AutoGen) excels in enterprise scenarios and multi-agent collaboration.

Other notable frameworks include **Pydantic AI** (valued for its type-safe design), **LlamaIndex** (focused on data-augmented agents), **Smolagents** (lightweight and experimental), and **AG2** (a successor to AutoGen). Many of these frameworks now support the **Model Context Protocol (MCP)** for standardized tool integrat

[non-fatal] Tracing client error 401. Response data is redacted.


### 2. The Planner Agent: given a question, comes up with a list of searches that should be made in Structured Outputs 'WebSearchItem'

Structured Outputs Allow an LLM to generate structured Python objects instead of plain text. This is commonly achieved using JSON Schema + Pydantic, where the LLM generates JSON that is converted into a validated Python object.

In [29]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")

    
class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [30]:
WebSearchPlan.model_json_schema()

{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'Your reasoning for why this search is important to the query.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search term to use for the web search.',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query.',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [31]:


INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent = Agent(
    name="Planner Agent",
    instructions=INSTRUCTIONS,
    output_type= WebSearchPlan,
    model=MODEL_NAME  
)


## 3. The Writer Agent: writes a robust report

In [32]:


INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 5-10 pages of content, at least 1000 words.
"""

class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")

writer_agent = Agent(
    name='Write Agent',
    instructions=INSTRUCTIONS,
    output_type=ReportData,
    model=MODEL_NAME
)


## Agent 4: The email agent

In [33]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email sent successfully"

In [34]:
send_email_tool.params_json_schema
send_email_tool.description

'Send out an email with the given subject and body to all sales prospects'

In [35]:
INSTRUCTIONS = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

email_agent = Agent(name="Email Agent", instructions=INSTRUCTIONS, tools=[send_email_tool], model=MODEL_NAME)

## Now to Orchestrate by Code

The next 2 functions will plan and execute the search, using the Agents, with calls to `Runner.run()`

In [ ]:
async def search(item: WebSearchItem):
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output


async def run_searches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches")
    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results


In [37]:
async def write_report(query: str, search_results: list[str]):
    print("Thinking about report...")
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output

async def send_report_email(report: ReportData):
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return result.final_output

In [38]:
query ="Most popular AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_results = await run_searches(query)
    report = await write_report(query, search_results)
    await send_report_email(report)  
    print("Hooray!")

Starting research...
Planning searches...


[non-fatal] Tracing client error 401. Response data is redacted.


Will perform 5 searches


[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.


websearch is active............ best AI agent frameworks for building autonomous agents 2026
websearch is active............ top AI agent frameworks comparison 2026
websearch is active............ AI agent frameworks enterprise adoption 2026


...

[non-fatal] Tracing client error 401. Response data is redacted.


Finished writing report
Writing email...


[non-fatal] Tracing client error 401. Response data is redacted.


Email sent
Hooray!


[non-fatal] Tracing client error 401. Response data is redacted.
